# Notebook 04 — IAM Test Set Evaluation
**D7047E Advanced Deep Learning | Group 14**

Evaluates **all available checkpoints** in `../checkpoints/` against the IAM test set.
Automatically picks up whatever models have been trained — no manual config needed.

- Binary: `best_binary_SimpleCNN.pth`
- Multiclass: `best_mc_SimpleCNN.pth`, `best_mc_ResNet_50.pth`, `best_mc_ViT_B_16.pth`

Run this after training completes to get final test metrics and comparison plots.

## 1. Configuration

In [ ]:
import sys
sys.path.insert(0, '..')
DATA_DIR       = '../dataset/iam_crossouts'
CHECKPOINT_DIR = '../checkpoints'
IMG_SIZE       = 224
BATCH_SIZE     = 64
NUM_WORKERS    = 8
from common import CATEGORIES
NC = len(CATEGORIES)
print(f'Config loaded. NC={NC}')

## 2. Setup

In [ ]:
!pip install -q torch torchvision pillow matplotlib scikit-learn wandb python-dotenv

In [ ]:
import os, gc
import torch, torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)
from dotenv import load_dotenv
import wandb

from common import (get_transforms, WANDB_PROJECT, CrossOutDataset, BinaryDataset,
                    CATEGORIES, rebuild_model, rebuild_binary_model)

load_dotenv()
wandb.login(key=os.environ.get('WANDB_API_KEY'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NC = len(CATEGORIES)
print(f'Device: {device}  |  {NC} classes: {CATEGORIES}')

## 3. Load Test Sets

In [ ]:
_, val_t = get_transforms(IMG_SIZE)
ldr_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
              pin_memory=True, prefetch_factor=2 if NUM_WORKERS > 0 else None)

mc_test_ds  = CrossOutDataset(os.path.join(DATA_DIR,'test','images'), CATEGORIES, val_t)
bin_test_ds = BinaryDataset(os.path.join(DATA_DIR,'test','images'), val_t)

mc_test_loader  = DataLoader(mc_test_ds,  shuffle=False, **ldr_kw)
bin_test_loader = DataLoader(bin_test_ds, shuffle=False, **ldr_kw)

print(f'Multiclass test: {len(mc_test_ds):,} samples')
print(f'Binary test:     {len(bin_test_ds):,} samples')

## 4. Evaluate All Multiclass Checkpoints

In [ ]:
mc_files = sorted([f for f in os.listdir(CHECKPOINT_DIR)
                   if f.startswith('best_mc_') and f.endswith('.pth')])
print(f'Found {len(mc_files)} multiclass checkpoints: {mc_files}\n')

mc_results = {}

for fname in mc_files:
    path = os.path.join(CHECKPOINT_DIR, fname)
    ckpt = torch.load(path, map_location=device)
    model_name = ckpt.get('model_name', fname.replace('best_mc_','').replace('.pth',''))

    model = rebuild_model(model_name, NC)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()

    preds, labels = [], []
    with torch.no_grad():
        for imgs, lbs in mc_test_loader:
            preds.extend(model(imgs.to(device)).argmax(1).cpu().tolist())
            labels.extend(lbs.tolist())

    acc    = accuracy_score(labels, preds)
    p      = precision_score(labels, preds, average='macro', zero_division=0)
    r      = recall_score(labels, preds, average='macro', zero_division=0)
    f1     = f1_score(labels, preds, average='macro', zero_division=0)
    epoch  = ckpt.get('epoch', '?')
    val_loss = ckpt.get('val_loss', float('nan'))

    mc_results[model_name] = {
        'acc': acc, 'precision': p, 'recall': r, 'f1': f1,
        'preds': preds, 'labels': labels,
        'best_epoch': epoch, 'best_val_loss': val_loss,
    }
    print(f'{model_name:<18}  Acc={acc:.4f}  P={p:.4f}  R={r:.4f}  Macro-F1={f1:.4f}  '
          f'(epoch={epoch}, val_loss={val_loss:.4f})')
    del model; torch.cuda.empty_cache(); gc.collect()

if mc_results:
    best_mc = max(mc_results, key=lambda k: mc_results[k]['f1'])
    print(f'\n★ Best multiclass: {best_mc}  Macro-F1={mc_results[best_mc]["f1"]:.4f}')

## 5. Evaluate Binary Checkpoint (SimpleCNN)

In [ ]:
bin_files = sorted([f for f in os.listdir(CHECKPOINT_DIR)
                    if f.startswith('best_binary_') and f.endswith('.pth')])
print(f'Found {len(bin_files)} binary checkpoints: {bin_files}\n')

bin_results = {}

for fname in bin_files:
    path = os.path.join(CHECKPOINT_DIR, fname)
    ckpt = torch.load(path, map_location=device)
    model_name = ckpt.get('model_name', fname.replace('best_binary_','').replace('.pth',''))

    model = rebuild_binary_model(model_name)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()

    preds, labels, probs = [], [], []
    with torch.no_grad():
        for imgs, lbs in bin_test_loader:
            out = model(imgs.to(device))
            probs.extend(torch.softmax(out, dim=1)[:,1].cpu().tolist())
            preds.extend(out.argmax(1).cpu().tolist())
            labels.extend(lbs.tolist())

    acc  = accuracy_score(labels, preds)
    p    = precision_score(labels, preds, zero_division=0)
    r    = recall_score(labels, preds, zero_division=0)
    f1   = f1_score(labels, preds, zero_division=0)
    auc  = roc_auc_score(labels, probs)
    epoch = ckpt.get('epoch', '?')
    val_loss = ckpt.get('val_loss', float('nan'))

    bin_results[model_name] = {
        'acc': acc, 'precision': p, 'recall': r, 'f1': f1, 'auc': auc,
        'preds': preds, 'labels': labels, 'probs': probs,
        'best_epoch': epoch, 'best_val_loss': val_loss,
    }
    print(f'{model_name:<18}  Acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f1:.4f}  AUC={auc:.4f}  '
          f'(epoch={epoch}, val_loss={val_loss:.4f})')
    del model; torch.cuda.empty_cache(); gc.collect()

if bin_results:
    best_bin = max(bin_results, key=lambda k: bin_results[k]['f1'])
    print(f'\n★ Best binary: {best_bin}  F1={bin_results[best_bin]["f1"]:.4f}')

## 6. Comparison Table

In [ ]:
print('=' * 75)
print('MULTICLASS — IAM Test Set Results')
print('=' * 75)
print(f'{"Model":<18} {"Accuracy":>10} {"Precision":>10} {"Recall":>8} {"Macro-F1":>10} {"Best Ep":>8}')
print('-' * 75)
for name, r in sorted(mc_results.items(), key=lambda x: -x[1]['f1']):
    marker = ' ★' if name == best_mc else ''
    print(f'{name:<18} {r["acc"]:>10.4f} {r["precision"]:>10.4f} {r["recall"]:>8.4f} '
          f'{r["f1"]:>10.4f} {str(r["best_epoch"]):>8}{marker}')

print()
print('=' * 75)
print('BINARY — IAM Test Set Results')
print('=' * 75)
print(f'{"Model":<18} {"Accuracy":>10} {"Precision":>10} {"Recall":>8} {"F1":>10} {"AUC":>8} {"Best Ep":>8}')
print('-' * 75)
for name, r in sorted(bin_results.items(), key=lambda x: -x[1]['f1']):
    marker = ' ★' if name == best_bin else ''
    print(f'{name:<18} {r["acc"]:>10.4f} {r["precision"]:>10.4f} {r["recall"]:>8.4f} '
          f'{r["f1"]:>10.4f} {r["auc"]:>8.4f} {str(r["best_epoch"]):>8}{marker}')

## 7. Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Multiclass
if mc_results:
    names   = sorted(mc_results.keys(), key=lambda k: -mc_results[k]['f1'])
    metrics = ['acc', 'precision', 'recall', 'f1']
    labels_m = ['Accuracy', 'Precision', 'Recall', 'Macro-F1']
    x = np.arange(len(names))
    w = 0.2
    colors = ['#4c72b0', '#55a868', '#c44e52', '#8172b2']
    for i, (metric, label, color) in enumerate(zip(metrics, labels_m, colors)):
        vals = [mc_results[n][metric] for n in names]
        axes[0].bar(x + i*w - 1.5*w, vals, w, label=label, color=color)
    axes[0].set_xticks(x); axes[0].set_xticklabels(names, rotation=15, ha='right')
    axes[0].set_ylim(0, 1.05); axes[0].set_title('Multiclass — IAM Test')
    axes[0].legend(fontsize=8); axes[0].set_ylabel('Score')

# Binary
if bin_results:
    names_b  = sorted(bin_results.keys(), key=lambda k: -bin_results[k]['f1'])
    metrics_b = ['acc', 'precision', 'recall', 'f1', 'auc']
    labels_b  = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']
    xb = np.arange(len(names_b))
    wb = 0.15
    colors_b = ['#4c72b0', '#55a868', '#c44e52', '#8172b2', '#ccb974']
    for i, (metric, label, color) in enumerate(zip(metrics_b, labels_b, colors_b)):
        vals = [bin_results[n][metric] for n in names_b]
        axes[1].bar(xb + i*wb - 2*wb, vals, wb, label=label, color=color)
    axes[1].set_xticks(xb); axes[1].set_xticklabels(names_b, rotation=15, ha='right')
    axes[1].set_ylim(0, 1.05); axes[1].set_title('Binary — IAM Test')
    axes[1].legend(fontsize=8); axes[1].set_ylabel('Score')

plt.suptitle('IAM Test Set — All Models Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('iam_test_comparison.png', dpi=150)
plt.show()
print('Saved: iam_test_comparison.png')

## 8. Confusion Matrices — All Multiclass Models

In [ ]:
if mc_results:
    n_models = len(mc_results)
    cols = min(n_models, 3)
    rows = (n_models + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*6, rows*5))
    axes = np.array(axes).flatten() if n_models > 1 else [axes]

    for i, name in enumerate(sorted(mc_results.keys(), key=lambda k: -mc_results[k]['f1'])):
        r  = mc_results[name]
        cm = confusion_matrix(r['labels'], r['preds'])
        ConfusionMatrixDisplay(cm, display_labels=CATEGORIES).plot(
            ax=axes[i], xticks_rotation=45, colorbar=False, cmap='Blues')
        marker = ' ★' if name == best_mc else ''
        axes[i].set_title(f'{name}{marker}\nAcc={r["acc"]:.3f}  Macro-F1={r["f1"]:.3f}')

    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Confusion Matrices — Multiclass IAM Test', fontsize=12)
    plt.tight_layout()
    plt.savefig('iam_test_mc_confusion.png', dpi=120)
    plt.show()
    print('Saved: iam_test_mc_confusion.png')

## 9. Classification Report — Best Multiclass Model

In [ ]:
if mc_results:
    r = mc_results[best_mc]
    print(f'Classification Report — {best_mc} (IAM Test)\n')
    print(classification_report(r['labels'], r['preds'],
                                target_names=CATEGORIES, zero_division=0))

## 10. Log to WandB

In [ ]:
for name, r in mc_results.items():
    run = wandb.init(project=WANDB_PROJECT, group='iam_test_eval',
                     name=f'mc_{name}', config=dict(model=name, task='multiclass'), reinit=True)
    wandb.log({'test_acc': r['acc'], 'test_precision': r['precision'],
               'test_recall': r['recall'], 'test_macro_f1': r['f1'],
               'best_epoch': r['best_epoch'], 'best_val_loss': r['best_val_loss']})
    run.finish()

for name, r in bin_results.items():
    run = wandb.init(project=WANDB_PROJECT, group='iam_test_eval',
                     name=f'bin_{name}', config=dict(model=name, task='binary'), reinit=True)
    wandb.log({'test_acc': r['acc'], 'test_precision': r['precision'],
               'test_recall': r['recall'], 'test_f1': r['f1'], 'test_auc': r['auc'],
               'best_epoch': r['best_epoch'], 'best_val_loss': r['best_val_loss']})
    run.finish()

print('All results logged to WandB.')